# Memory

模型本身是不会记忆任何上下文的，只能依靠用户本身的输入去产生输出。


Memory，是LangChain中用于多轮对话中保存和管理上下文信息（比如文本、图像、音频等）的组件。它让应用能够记住用户之前说了什么，从而实现对话的上下文感知能力 ，为构建真正智能和上下文感知的链式对话系统提供了基础。

### 不使用memory模块 在代码里实现记忆功能

In [22]:
# 1、获取大模型
import os
import dotenv
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

In [23]:
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages.ai import AIMessage


def chat_with_model(answer):
    # 2、提供提示词模板：ChatPromptTemplate
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "你是一个人工智能的助手"),
        ("human", "{question}")
    ])

    while True:
        # 3、获取chain，并调用大模型得到响应
        chain = prompt_template | llm
        response = chain.invoke({"question": answer})

        # 4、输出大模型的响应
        print(f"模型回复：{response.content}")

        # 5、继续获取用户的问题
        user_input = input("你还有其他问题吗？(输入'退出'时，结束会话)")

        # 6、指明退出循环的方式
        if (user_input == "退出"):
            break

        # 7、将上述新生成的消息存放到提示词模板的消息列表中
        prompt_template.messages.append(AIMessage(content=response.content))
        prompt_template.messages.append(HumanMessage(content=user_input))


chat_with_model("你好，很高兴认识你！")

模型回复：你好！我也很高兴认识你！有什么我可以帮助你的吗？
模型回复：好的，你可以叫我小智！有什么问题或者需要帮助的地方吗？
模型回复：我叫小智！你可以叫我这个名字，有什么我可以帮你的吗？


### Memory 模块的使用

Memory模块的设计：

层次1(最直接的方式)：保留一个聊天消息列表

层次2(简单的新思路)：只返回最近交互的k条消息

层次3(稍微复杂一点)：返回过去k条消息的简洁摘要

层次4(更复杂)：从存储的消息中提取实体，并且仅返回有关当前运行中引用的实体的信息

#### ChatMessageHistory

最基础、最核心的，底层存储工具；API文档中名为InMemoryChatMessageHistory

场景1：记忆存储

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
# from langchain.memory import ChatMessageHistory
from langchain_core.prompts import PromptTemplate

# 1、InMemoryChatMessageHistory的实例化

history = InMemoryChatMessageHistory()

# 2、添加相关的消息进行存储

history.add_user_message("你好")

history.add_ai_message("很高兴认识你")

# 3、打印存储的消息
print(history.messages)

[HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


场景2：对接大模型

In [29]:
# 1、获取大模型
import os
import dotenv
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

In [35]:
from langchain_core.chat_history import InMemoryChatMessageHistory


# 1、InMemoryChatMessageHistory的实例化

history = InMemoryChatMessageHistory()

# 2、添加相关的消息进行存储
history.add_user_message("你好")
history.add_ai_message("很高兴认识你")
history.add_user_message("帮我计算1 + 2 * 3 = ？")

response = llm.invoke(history.messages)
print(response.content)

根据运算优先级，先进行乘法运算，然后再进行加法运算：

1 + 2 * 3 = 1 + 6 = 7

所以，1 + 2 * 3 = 7。


#### 2、ConversationBufferMemory的使用

按照原始顺序存储完整的对话历史

适用于对话轮次少、依赖完整上下文的场景

举例1：以字符串的方式返回存储的信息

In [ ]:
from langchain_classic.memory import ConversationBufferMemory
# 从classic包调用
# 1、ConversationBufferMemory的实例化
memory = ConversationBufferMemory()

# 2、存储相关的消息
# inputs对应的就是用户消息human，outputs对应的就是ai消息
memory.save_context(inputs={"human": "你好，我叫小明"}, outputs={"ai": "很高兴认识你"})
memory.save_context(inputs={"input": "帮我回答一下1+2*3=?"}, outputs={"output": "7"})

# 3、获取存储的信息
print(memory.load_memory_variables({}))
# 返回纯文本字符串
#说明：返回的字典结构的key叫history.

{'history': 'Human: 你好，我叫小明\nAI: 很高兴认识你\nHuman: 帮我回答一下1+2*3=?\nAI: 7'}


C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\1768580331.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


举例2：以消息列表的方式返回存储的信息

In [48]:
from langchain_classic.memory import ConversationBufferMemory

# 1、ConversationBufferMemory的实例化
memory = ConversationBufferMemory(return_messages=True)

# 2、存储相关的消息
# inputs对应的就是用户消息，outputs对应的就是ai消息
memory.save_context(inputs={"human": "你好，我叫小明"}, outputs={"ai": "很高兴认识你"})
memory.save_context(inputs={"input": "帮我回答一下1+2*3=?"}, outputs={"output": "7"})

# 3、获取存储的信息
#返回消息列表的方式1：
print(memory.load_memory_variables({}))

print("\n")

#返回消息列表的方式2：
print(memory.chat_memory.messages)

#说明：返回的字典结构的key叫history.

{'history': [HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下1+2*3=?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={})]}


[HumanMessage(content='你好，我叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='帮我回答一下1+2*3=?', additional_kwargs={}, response_metadata={}), AIMessage(content='7', additional_kwargs={}, response_metadata={})]


举例3：结合大模型、提示词模板的使用（PromptTemplate）

In [51]:
from langchain_classic.chains.llm import LLMChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

当前对话历史: {history}

人类问题: {question}

回复:
"""
)

# 3、提供memory实例
memory = ConversationBufferMemory()

# 4、提供Chain
chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

response = chain.invoke({"question": "你好，我的名字叫小明"})
print(response)

C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\2847977784.py:25: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)


{'question': '你好，我的名字叫小明', 'history': '', 'text': '你好，小明！很高兴认识你。有什么我可以帮你的吗？'}


In [52]:
response = chain.invoke({"question": "我叫什么名字呢？"})
print(response)

{'question': '我叫什么名字呢？', 'history': 'Human: 你好，我的名字叫小明\nAI: 你好，小明！很高兴认识你。有什么我可以帮你的吗？', 'text': '你叫小明。请问还有其他我可以帮你解答的事情吗？'}


举例4：结合大模型、提示词模板的使用（ChatPromptTemplate）

In [ ]:
# 1.导入相关包
from langchain_core.messages import SystemMessage
from langchain_classic.chains.llm import LLMChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder,ChatPromptTemplate,HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI


# 2.创建LLM
llm = ChatOpenAI(model_name='gpt-4o-mini')

# 3.创建Prompt  写成消息列表的形式
prompt = ChatPromptTemplate.from_messages([
    ("system","你是一个与人类对话的机器人。"),
    MessagesPlaceholder(variable_name='history'),
    ("human","问题：{question}")
])

# 4.创建Memory
memory = ConversationBufferMemory(return_messages=True)
# 5.创建LLMChain
llm_chain = LLMChain(prompt=prompt,llm=llm, memory=memory)

# 6.调用LLMChain
res1 = llm_chain.invoke({"question": "中国首都在哪里？"})
print(res1,end="\n\n")

#答复的同时把消息放到history里

{'question': '中国首都在哪里？', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都是北京。', additional_kwargs={}, response_metadata={})], 'text': '中国的首都是北京。'}



In [54]:
res2 = llm_chain.invoke({"question": "我刚刚问了什么"})
print(res2)

{'question': '我刚刚问了什么', 'history': [HumanMessage(content='中国首都在哪里？', additional_kwargs={}, response_metadata={}), AIMessage(content='中国的首都是北京。', additional_kwargs={}, response_metadata={}), HumanMessage(content='我刚刚问了什么', additional_kwargs={}, response_metadata={}), AIMessage(content='你问了“中国首都在哪里？”', additional_kwargs={}, response_metadata={})], 'text': '你问了“中国首都在哪里？”'}


#### 3、ConversationChain的使用

举例1：以PromptTemplate为例

In [56]:
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
prompt_template = PromptTemplate.from_template(
    template="""
    你可以与人类对话。

当前对话历史: {history}

人类问题: {input}

回复:
"""
)

# # 3、提供memory实例
# memory = ConversationBufferMemory()
#
# # 4、提供Chain
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

# 3、创建ConversationChain的实例
chain = ConversationChain(llm = llm, prompt=prompt_template)

response = chain.invoke({"input": "你好，我的名字叫小明"})
print(response)

C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\2918950995.py:29: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  chain = ConversationChain(llm = llm, prompt=prompt_template)


{'input': '你好，我的名字叫小明', 'history': '', 'response': '你好，小明！很高兴认识你！有什么我可以帮助你的吗？'}


In [57]:
response = chain.invoke({"input": "我的名字叫什么？"})
print(response)

{'input': '我的名字叫什么？', 'history': 'Human: 你好，我的名字叫小明\nAI: 你好，小明！很高兴认识你！有什么我可以帮助你的吗？', 'response': '你的名字叫小明。'}


举例2：使用默认提供的提示词模板

In [59]:
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_openai import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate

# 1、创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 2、提供提示词模板
# prompt_template = PromptTemplate.from_template(
#     template="""
#     你可以与人类对话。
#
# 当前对话历史: {history}
#
# 人类问题: {input}
#
# 回复:
# """
# )

# # 3、提供memory实例
# memory = ConversationBufferMemory()
#
# # 4、提供Chain
# chain = LLMChain(llm=llm, prompt=prompt_template, memory=memory)

# 3、创建ConversationChain的实例（内部提供了默认的提示词模板。而此模板中的变量是{input}、{history}
chain = ConversationChain(llm = llm)

response = chain.invoke({"input": "你好，我的名字叫小明"})
print(response)

{'input': '你好，我的名字叫小明', 'history': '', 'response': '你好，小明！很高兴认识你！我叫AI助手。你今天过得怎么样？有什么想聊的话题吗？'}


In [60]:
response = chain.invoke({"input": "我的名字叫什么？"})
print(response)

{'input': '我的名字叫什么？', 'history': 'Human: 你好，我的名字叫小明\nAI: 你好，小明！很高兴认识你！我叫AI助手。你今天过得怎么样？有什么想聊的话题吗？', 'response': '你的名字叫小明！你有没有什么特别的事情想分享，或者想聊聊你喜欢的活动、美食等？'}


#### 4、ConversationBufferWindowMemory的使用

只返回最近交互的k条消息

ConversationBufferMemory无限地将历史对话信息填充到history中，导致内存量非常大，且消耗的token多。

举例1：

In [61]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferWindowMemory

# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=1)
# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手"})
memory.save_context({"input": "你的生日是哪天？"}, {"output": "我不清楚"})
# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

{'history': 'Human: 你的生日是哪天？\nAI: 我不清楚'}


C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\548428214.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=1)


举例2：返回消息构成的上下文记忆

In [63]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferWindowMemory

# 2.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=2, return_messages=True)
# 3.保存消息
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手小智"})
memory.save_context({"input": "初次对话，你能介绍一下你自己吗？"}, {"output": "当然可以了。我是一个无所不能的小智。"})
# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

{'history': [HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}), AIMessage(content='我是AI助手小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='初次对话，你能介绍一下你自己吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='当然可以了。我是一个无所不能的小智。', additional_kwargs={}, response_metadata={})]}


举例3：结合llm、chain的使用

In [64]:
from langchain_classic.memory import ConversationBufferWindowMemory
# 1.导入相关包
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains.llm import LLMChain

# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会表示不知道。

当前对话：
{history}
Human: {question}
AI:"""

# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)

# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=1)

# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    #verbose=True,
)

# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question":"你好，我是孙小空"})
print(respon1)
# 8.执行链（第二次提问）
respon2 =conversation_with_summary.invoke({"question":"我还有两个师弟，一个是猪小戒，一个是沙小僧"})
print(respon2)
# 9.执行链（第三次提问）
respon3 =conversation_with_summary.invoke({"question":"我今年高考，竟然考上了1本"})
print(respon3)
# 10.执行链（第四次提问）
respon4 =conversation_with_summary.invoke({"question":"我叫什么名字？"})
print(respon4)

{'question': '你好，我是孙小空', 'history': '', 'text': '你好，孙小空！很高兴认识你。有什么我可以帮助你的吗？'}
{'question': '我还有两个师弟，一个是猪小戒，一个是沙小僧', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴认识你。有什么我可以帮助你的吗？', 'text': '很高兴认识你的师弟们，猪小戒和沙小僧！他们的名字听起来很有趣。你们在一起都做些什么呢？或者说说你们的故事吧！'}
{'question': '我今年高考，竟然考上了1本', 'history': 'Human: 我还有两个师弟，一个是猪小戒，一个是沙小僧\nAI: 很高兴认识你的师弟们，猪小戒和沙小僧！他们的名字听起来很有趣。你们在一起都做些什么呢？或者说说你们的故事吧！', 'text': '太棒了，恭喜你考上了本科！这是一个值得庆祝的成就。你有没有选择好专业或者未来的计划呢？高考期间有没有什么特别的经历或者挑战？'}
{'question': '我叫什么名字？', 'history': 'Human: 我今年高考，竟然考上了1本\nAI: 太棒了，恭喜你考上了本科！这是一个值得庆祝的成就。你有没有选择好专业或者未来的计划呢？高考期间有没有什么特别的经历或者挑战？', 'text': '我不知道你叫什么名字，但很乐意称呼你为朋友！如果你愿意，可以告诉我你的名字，或者分享一些关于你自己的事情。你对即将开始的大学生活有什么期待吗？'}


举例4：修改举例3中的参数k

In [65]:
from langchain_classic.memory import ConversationBufferWindowMemory
# 1.导入相关包
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains.llm import LLMChain

# 2.定义模版
template = """以下是人类与AI之间的友好对话描述。AI表现得很健谈，并提供了大量来自其上下文的具体细节。如果AI不知道问题的答案，它会表示不知道。

当前对话：
{history}
Human: {question}
AI:"""

# 3.定义提示词模版
prompt_template = PromptTemplate.from_template(template)

# 4.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 5.实例化ConversationBufferWindowMemory对象，设定窗口阈值
memory = ConversationBufferWindowMemory(k=3)

# 6.定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    #verbose=True,
)

# 7.执行链（第一次提问）
respon1 = conversation_with_summary.invoke({"question":"你好，我是孙小空"})
print(respon1)
# 8.执行链（第二次提问）
respon2 =conversation_with_summary.invoke({"question":"我还有两个师弟，一个是猪小戒，一个是沙小僧"})
print(respon2)
# 9.执行链（第三次提问）
respon3 =conversation_with_summary.invoke({"question":"我今年高考，竟然考上了1本"})
print(respon3)
# 10.执行链（第四次提问）
respon4 =conversation_with_summary.invoke({"question":"我叫什么名字？"})
print(respon4)

{'question': '你好，我是孙小空', 'history': '', 'text': '你好，孙小空！很高兴认识你！你今天过得怎么样？有什么我可以帮助你的吗？'}
{'question': '我还有两个师弟，一个是猪小戒，一个是沙小僧', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴认识你！你今天过得怎么样？有什么我可以帮助你的吗？', 'text': '哇，听起来你们的师门很有意思！猪小戒和沙小僧这两个名字让我想起了《西游记》。你们是不是也在追求某种目标或经历冒险呢？能跟我分享一下你们的故事吗？'}
{'question': '我今年高考，竟然考上了1本', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴认识你！你今天过得怎么样？有什么我可以帮助你的吗？\nHuman: 我还有两个师弟，一个是猪小戒，一个是沙小僧\nAI: 哇，听起来你们的师门很有意思！猪小戒和沙小僧这两个名字让我想起了《西游记》。你们是不是也在追求某种目标或经历冒险呢？能跟我分享一下你们的故事吗？', 'text': '太棒了，恭喜你考上了一本大学！这是一个了不起的成就，你一定付出了很多努力。你已经决定去哪个学校吗？或者你对未来的专业有什么打算呢？'}
{'question': '我叫什么名字？', 'history': 'Human: 你好，我是孙小空\nAI: 你好，孙小空！很高兴认识你！你今天过得怎么样？有什么我可以帮助你的吗？\nHuman: 我还有两个师弟，一个是猪小戒，一个是沙小僧\nAI: 哇，听起来你们的师门很有意思！猪小戒和沙小僧这两个名字让我想起了《西游记》。你们是不是也在追求某种目标或经历冒险呢？能跟我分享一下你们的故事吗？\nHuman: 我今年高考，竟然考上了1本\nAI: 太棒了，恭喜你考上了一本大学！这是一个了不起的成就，你一定付出了很多努力。你已经决定去哪个学校吗？或者你对未来的专业有什么打算呢？', 'text': '你刚刚告诉我你叫孙小空。你还想聊些什么呢？'}


#### 5、ConversationTokenBufferMemory的使用

Token数量限制，如果字符数量超出指定数目，切掉这个对话的早期部分，保留最近的交流相应的字符数量。

举例1：

In [ ]:
# 1.导入相关包
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI

# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义ConversationTokenBufferMemory对象
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=20  # 设置token上限，默认值为2000
)

# 添加对话
memory.save_context({"input": "你好吗？"}, {"output": "我很好，谢谢！"})
memory.save_context({"input": "今天天气如何？"}, {"output": "晴天，25度"})
#并不保留消息对，不建议使用

# 查看当前记忆
print(memory.load_memory_variables({}))

C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\2235930964.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationTokenBufferMemory(


{'history': 'AI: 晴天，25度'}


#### 2、ConversationSummaryMemory的使用

全部内容保存下来过于浪费，按照对话条数/token截断无法保证节省内存又保证对话质量；
智能压缩对话历史

举例1：

如果实例化ConversationSummaryMemory前，没有历史消息，可以使用构造方法实例化

In [ ]:
# 1.导入相关包
from langchain_classic.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain_openai import ChatOpenAI

# 2.创建大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.定义ConversationSummaryMemory对象
memory = ConversationSummaryMemory(llm=llm)

# 4.存储消息  创建的时候没有历史消息，可以构造
memory.save_context({"input": "你好"}, {"output": "怎么了"})
memory.save_context({"input": "你是谁"}, {"output": "我是AI助手小智"})
memory.save_context({"input": "初次对话，你能介绍一下你自己吗？"}, {"output": "当然可以了。我是一个无所不能的小智。"})

# 5.读取消息（总结后的）
print(memory.load_memory_variables({}))

C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\2722698371.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm)


{'history': 'The human greets the AI with "你好" (hello), and the AI responds by asking, "怎么了" (what\'s wrong). The human then asks, "你是谁" (who are you), to which the AI replies, "我是AI助手小智" (I am the AI assistant Xiao Zhi). The human asks for more information about the AI, and the AI responds that it is an all-capable assistant named Xiao Zhi.'}


举例2：如果实例化ConversationSummaryMemory前，已经有历史消息，可以调用from_messages()实例化

In [71]:
# 1.导入相关包
from langchain_classic.memory import ConversationSummaryMemory, ChatMessageHistory
from langchain_openai import ChatOpenAI

# 2.定义ChatMessageHistory对象
llm = ChatOpenAI(model="gpt-4o-mini")

# 3.假设原始消息
history = ChatMessageHistory()
history.add_user_message("你好，你是谁？")
history.add_ai_message("我是AI助手小智")

# 4.创建ConversationSummaryMemory的实例
memory = ConversationSummaryMemory.from_messages(
    llm = llm,
    #是生成摘要的原材料 保留完整对话供必要时回溯。当新增对话时，LLM需要结合原始历史生成新摘要
    chat_memory = history,
)

print(memory.load_memory_variables({}))


memory.save_context(inputs={"human":"我的名字叫小明"},outputs={"ai":"很高兴认识你"})

print(memory.load_memory_variables({}))

#记录了历史的交互的消息
print(memory.chat_memory.messages)


{'history': 'The human greets the AI and asks who it is. The AI responds that it is the AI assistant named Xiao Zhi.'}
{'history': 'The human greets the AI and asks who it is. The AI responds that it is the AI assistant named Xiao Zhi. The human introduces himself as Xiao Ming, and the AI expresses pleasure in meeting him.'}
[HumanMessage(content='你好，你是谁？', additional_kwargs={}, response_metadata={}), AIMessage(content='我是AI助手小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


#### 3、ConversationSummaryBufferMemory的使用

混合型记忆机制，保留最近的原始记录，并对较早期的对话内容进行智能摘要。

举例1：

In [ ]:
from langchain_classic.memory import ConversationSummaryBufferMemory

# 获取大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 实例化ConversationSummaryBufferMemory
memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=40,  #控制缓冲区的大小  保留的消息
    return_messages=True,
)

# 向memory中存储信息 从第二个output开始
memory.save_context(inputs={"input":"你好，我的名字叫小明"},outputs={"output":"很高兴认识你"})
memory.save_context(inputs={"input":"李白是哪个朝代的诗人"},outputs={"output":"李白是唐朝的诗人"})
memory.save_context(inputs={"input":"唐宋八大家里有苏轼吗？"},outputs={"output":"有"})

print(memory.load_memory_variables({}))

print("\n")

print(memory.chat_memory.messages)

C:\Users\yadddd\AppData\Local\Temp\ipykernel_62664\2434037194.py:7: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(


{'history': [SystemMessage(content='The human introduces themselves as 小明. The AI responds that it is pleased to meet them. The human then asks which dynasty the poet Li Bai belongs to.', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝的诗人', additional_kwargs={}, response_metadata={}), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={})]}


[AIMessage(content='李白是唐朝的诗人', additional_kwargs={}, response_metadata={}), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={})]


对照组

In [73]:
from langchain_classic.memory import ConversationSummaryBufferMemory

# 获取大模型
llm = ChatOpenAI(model="gpt-4o-mini")

# 实例化ConversationSummaryBufferMemory
memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=100,  #控制缓冲区的大小
    return_messages=True,
)

# 向memory中存储信息
memory.save_context(inputs={"input":"你好，我的名字叫小明"},outputs={"output":"很高兴认识你"})
memory.save_context(inputs={"input":"李白是哪个朝代的诗人"},outputs={"output":"李白是唐朝的诗人"})
memory.save_context(inputs={"input":"唐宋八大家里有苏轼吗？"},outputs={"output":"有"})

print(memory.load_memory_variables({}))

print("\n")

print(memory.chat_memory.messages)

{'history': [HumanMessage(content='你好，我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='李白是哪个朝代的诗人', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝的诗人', additional_kwargs={}, response_metadata={}), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={})]}


[HumanMessage(content='你好，我的名字叫小明', additional_kwargs={}, response_metadata={}), AIMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={}), HumanMessage(content='李白是哪个朝代的诗人', additional_kwargs={}, response_metadata={}), AIMessage(content='李白是唐朝的诗人', additional_kwargs={}, response_metadata={}), HumanMessage(content='唐宋八大家里有苏轼吗？', additional_kwargs={}, response_metadata={}), AIMessage(content='有', additional_kwargs={}, response_metadata={})]


举例2：模拟客服交互

In [74]:
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.llm import LLMChain

# 1、初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500
)

# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=400,
    memory_key="chat_history",
    return_messages=True
)

# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
)

# 5、模拟多轮对话
dialogue = [
    ("你好，我想查询订单12345的状态", None),
    ("这个订单是上周五下的", None),
    ("我现在急着用，能加急处理吗", None),
    ("等等，我可能记错订单号了，应该是12346", None),
    ("对了，你们退货政策是怎样的", None)
]

# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")

# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

用户: 你好，我想查询订单12345的状态
客服: 你好！感谢你联系我。关于订单12345的状态，我需要查一下。请稍等片刻，我会尽快为你提供最新信息。

用户: 这个订单是上周五下的
客服: 谢谢你提供的信息！让我来查一下上周五下的订单12345的状态。请稍等片刻，我会尽快给你回复。

用户: 我现在急着用，能加急处理吗
客服: 我理解你的着急，感谢你的耐心。关于加急处理订单的请求，我会尽量帮你协调。请你稍等，我会向相关部门反馈你的需求，并尽快给你回复。谢谢你的理解！

用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题！谢谢你更新订单号。我现在来查询订单12346的状态。请稍等片刻，我会尽快为你提供信息。

用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策如下：

1. **退货期限**：大部分商品在收到后7天内可以申请退货。
2. **商品状态**：退货的商品需要保持未使用状态，并且包装完好，包括所有配件、说明书等。
3. **申请流程**：请在我们的官网或APP上提交退货申请，填写相关信息并选择退货原因。
4. **运费**：符合退货政策的商品，运费由我们承担；如因个人原因退货，运费需由客户自行承担。

如果你有具体的商品或订单需要退货，可以告诉我，我会提供更详细的指导！


=== 当前记忆内容 ===
{'chat_history': [HumanMessage(content='你好，我想查询订单12345的状态', additional_kwargs={}, response_metadata={}), AIMessage(content='你好！感谢你联系我。关于订单12345的状态，我需要查一下。请稍等片刻，我会尽快为你提供最新信息。', additional_kwargs={}, response_metadata={}), HumanMessage(content='这个订单是上周五下的', additional_kwargs={}, response_metadata={}), AIMessage(content='谢谢你提供的信息！让我来查一下上周五下的订单12345的状态。请稍等片刻，我会尽快给你回复。', additional_kwargs={}, response_metadata={}), HumanMes

对照组

In [76]:
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.llm import LLMChain

# 1、初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500
)

# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),  #history的信息存储
    ("human", "{input}")
])


# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,
    memory_key="chat_history",
    return_messages=True
)

# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
)

# 5、模拟多轮对话
dialogue = [
    ("你好，我想查询订单12345的状态", None),
    ("这个订单是上周五下的", None),
    ("我现在急着用，能加急处理吗", None),
    ("等等，我可能记错订单号了，应该是12346", None),
    ("对了，你们退货政策是怎样的", None)
]

# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")

# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

用户: 你好，我想查询订单12345的状态
客服: 您好！感谢您联系我。关于订单12345的状态，我会为您查询一下。请稍等片刻。 

（此处可以根据系统查询结果回复用户，例如：） 

您的订单12345目前已发货，预计将在3个工作日内送达。如有其他问题，欢迎随时询问！

用户: 这个订单是上周五下的
客服: 感谢您提供的信息！上周五下的订单12345通常会在1-3个工作日内处理并发货。请您再耐心等候一下，如果您希望我进一步查询具体的发货状态，请告诉我，我会尽快为您确认！如有其他问题，随时欢迎您询问。

用户: 我现在急着用，能加急处理吗
客服: 我理解您急需这个订单的心情。关于加急处理的请求，通常需要您直接联系商家的客服团队，他们会根据具体情况为您提供帮助。不过，我会尽力为您查询是否可以加急处理，稍等一下。感谢您的耐心！

用户: 等等，我可能记错订单号了，应该是12346
客服: 没问题，我会为您查询订单号12346的状态。请稍等片刻。谢谢您的耐心！

用户: 对了，你们退货政策是怎样的
客服: 我们的退货政策是这样的：如果您对购买的商品不满意，可以在收到商品后的30天内申请退货。商品需要保持未使用状态，并且包装完好。退货申请可以通过我们的官方网站进行，具体步骤会在申请过程中详细说明。 

如果您有任何其他问题或需要进一步的帮助，请随时告诉我！


=== 当前记忆内容 ===
{'chat_history': [SystemMessage(content="The human asks the AI to check the status of order 12345. The AI acknowledges the request and states it will check the status. After a brief wait, the AI informs the human that order 12345 has been shipped and is expected to arrive within three business days, inviting further questions if needed. The human provides additional information that the order was 